**The section below is for cleaning the CVE, outbreak, phr, enrollment for K12 students at county level**

In [1]:
library(tidyverse)
library(dplyr)
outbreak <- read.csv("data/count.csv")
cve <- read.csv("data/cve.csv")
phr_conv <- read.csv("data/brfss/phr_conversion.csv")
df <- cve %>%
  inner_join(outbreak, by = "County",
             suffix = c("_cve", "_out"))
head(df)

Warning message:
"Paket 'forcats' wurde unter R Version 4.4.1 erstellt"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.1     v stringr   1.5.1
v ggplot2   3.5.1     v tibble    3.2.1
v lubridate 1.9.3     v tidyr     1.3.1
v purrr     1.0.2     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


,County,X2016_cve,X2017_cve,X2018_cve,X2019_cve,X2020_cve,X2021_cve,X2022_cve,X2023_cve,X2024_cve,...,X2016_out,X2017_out,X2018_out,X2019_out,X2020_out,X2021_out,X2022_out,X2023_out,X2024_out,X2025_out
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,...,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,Anderson,0.35%,0.50%,0.75%,0.94%,1.00%,1.11%,1.62%,1.64%,2.05%,...,0,0,0,0,0,0,0,0,0,0
2,Andrews,0.75%,0.90%,1.07%,1.50%,0.39%,1.36%,0.45%,1.57%,1.43%,...,0,0,0,0,0,0,0,0,0,3
3,Angelina,0.70%,0.63%,0.67%,0.82%,0.96%,0.86%,1.03%,1.53%,1.98%,...,0,0,0,0,0,0,0,0,0,0
4,Aransas,1.22%,1.39%,1.49%,1.71%,1.57%,1.51%,0.00%,1.91%,1.91%,...,0,0,0,0,0,0,0,0,0,0
5,Archer,0.61%,0.45%,0.81%,1.43%,1.29%,1.56%,1.68%,2.32%,2.80%,...,0,0,0,0,0,0,0,0,0,0
6,Armstrong,0.59%,1.90%,1.44%,2.35%,3.53%,3.53%,3.77%,4.46%,4.95%,...,0,0,0,0,0,0,0,0,0,0


In [2]:
df2025 <- df %>%
  transmute(
    County,
    cve = `X2025_cve`,
    outbreak = `X2025_out`
  )
head(df2025)

,County,cve,outbreak
,<chr>,<chr>,<int>
1,Anderson,2.54%,0
2,Andrews,1.91%,3
3,Angelina,2.50%,0
4,Aransas,2.06%,0
5,Archer,2.70%,0
6,Armstrong,5.24%,0


In [3]:
df2025$cve <- gsub("%", "", df2025$cve)
df2025$cve <- as.numeric(df2025$cve)
head(df2025)
summary(df2025)

,County,cve,outbreak
,<chr>,<dbl>,<int>
1,Anderson,2.54,0
2,Andrews,1.91,3
3,Angelina,2.50,0
4,Aransas,2.06,0
5,Archer,2.70,0
6,Armstrong,5.24,0


    County               cve            outbreak      
 Length:254         Min.   : 0.000   Min.   :  0.000  
 Class :character   1st Qu.: 1.492   1st Qu.:  0.000  
 Mode  :character   Median : 2.490   Median :  0.000  
                    Mean   : 2.817   Mean   :  2.925  
                    3rd Qu.: 3.765   3rd Qu.:  0.000  
                    Max.   :14.540   Max.   :414.000  

In [4]:
enrollment <- read.csv("data/enrollment.csv")

enrollment <- enrollment %>%
  mutate(
    County = County %>%
      str_to_lower() %>%
      str_remove(" county") %>%
      str_trim() %>%
      str_to_title()
  )
enrollment <- enrollment[enrollment$County != "Grand Total", ]
View(enrollment)

,County,Enrollment.Sum
,<chr>,<int>
1,Anderson,7808
2,Andrews,4209
3,Angelina,15649
4,Aransas,2913
5,Archer,2110
6,Armstrong,297
7,Atascosa,9046
8,Austin,6290
9,Bailey,1330


In [5]:
df2025 <- df2025 %>%
  left_join(enrollment, by = "County") %>%
  transmute(
    County,
    cve,
    outbreak,
    enrollment = Enrollment.Sum
  )


In [6]:
head(df2025)

,County,cve,outbreak,enrollment
,<chr>,<dbl>,<int>,<int>
1,Anderson,2.54,0,7808
2,Andrews,1.91,3,4209
3,Angelina,2.50,0,15649
4,Aransas,2.06,0,2913
5,Archer,2.70,0,2110
6,Armstrong,5.24,0,297


In [7]:
df2025 <- df2025 %>%
  inner_join(phr_conv, by = "County") %>%
    transmute(
      County,
      cve,
      outbreak,
      enrollment,
      phr = PHR
    )
head(df2025)
summary(df2025)

,County,cve,outbreak,enrollment,phr
,<chr>,<dbl>,<int>,<int>,<int>
1,Anderson,2.54,0,7808,4
2,Andrews,1.91,3,4209,9
3,Angelina,2.50,0,15649,5
4,Aransas,2.06,0,2913,11
5,Archer,2.70,0,2110,2
6,Armstrong,5.24,0,297,1


    County               cve            outbreak         enrollment    
 Length:254         Min.   : 0.000   Min.   :  0.000   Min.   :    93  
 Class :character   1st Qu.: 1.492   1st Qu.:  0.000   1st Qu.:  1180  
 Mode  :character   Median : 2.490   Median :  0.000   Median :  3240  
                    Mean   : 2.817   Mean   :  2.925   Mean   : 22023  
                    3rd Qu.: 3.765   3rd Qu.:  0.000   3rd Qu.: 10549  
                    Max.   :14.540   Max.   :414.000   Max.   :879002  
                                                       NA's   :5       
      phr        
 Min.   : 1.000  
 1st Qu.: 2.000  
 Median : 5.000  
 Mean   : 5.323  
 3rd Qu.: 8.000  
 Max.   :11.000  
                 

In [8]:
write.csv(df2025, file = "data/county_data.csv", row.names = FALSE)

**This section is for cleaning the general census data in texas for each county up to need**

In [9]:
# ----- Pull all county-level variables for Texas -----
library(viridis)
library(tidycensus)
census_api_key("d57d54c0ecada0a165fe2386aa8f5078acb68d5b")
texas_counties <- get_acs(
  geography = "county",
  state     = "TX",
  year      = 2024,       # or whichever year matches your exemption data
  survey    = "acs5",     # 5-year ACS = more reliable for small counties
  variables = c(
    # Demographics
    pct_hispanic     = "DP05_0071P",  # % Hispanic
    pct_black        = "DP05_0078P",  # % Black non-Hispanic
    pct_white        = "DP05_0077P",  # % White non-Hispanic
    
    # Socioeconomic
    pct_poverty      = "DP03_0128P",  # % below poverty line
    pct_uninsured    = "DP03_0099P",  # % no health insurance
    pct_college      = "DP02_0068P",  # % college degree+
    median_income    = "DP03_0062",   # median household income

    
    # Rural/accessx
    pct_foreign_born = "DP02_0093P"   # % foreign born
  ),
  output = "wide"   # puts each variable in its own column
)

# Clean it up
texas_counties <- texas_counties %>%
  # Extract county FIPS code
  mutate(county_fips = substr(GEOID, 3, 5)) %>%
  # Keep only the estimate columns (drop margin of error)
  select(GEOID, NAME, county_fips,
         pct_hispanicE, pct_blackE, pct_whiteE,
         pct_povertyE, pct_uninsuredE, 
         pct_collegeE, median_incomeE,
         pct_foreign_bornE) %>%
  # Rename to cleaner names
  rename(
    pct_hispanic     = pct_hispanicE,
    pct_black        = pct_blackE,
    pct_white        = pct_whiteE,
    pct_poverty      = pct_povertyE,
    pct_uninsured    = pct_uninsuredE,
    pct_college      = pct_collegeE,
    median_income    = median_incomeE,
    pct_foreign_born = pct_foreign_bornE
  )

head(texas_counties)
summary(texas_counties)

Lade n"otiges Paket: viridisLite

Warning message:
"Paket 'tidycensus' wurde unter R Version 4.4.3 erstellt"
To install your API key for use in future sessions, run this function with `install = TRUE`.

Getting data from the 2020-2024 5-year ACS

Using the ACS Data Profile



GEOID,NAME,county_fips,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
48001,"Anderson County, Texas",001,0.1,0.3,2.6,17.4,18.5,15.4,62068,1.1
48003,"Andrews County, Texas",003,0.0,0.1,0.9,15.6,22.4,17.7,72242,1.2
48005,"Angelina County, Texas",005,0.0,0.2,3.1,15.8,17.7,17.9,60960,1.2
48007,"Aransas County, Texas",007,0.0,0.7,0.9,14.2,12.8,28.8,69466,1.5
48009,"Archer County, Texas",009,0.0,0.0,1.5,9.2,13.9,25.4,72159,0.8
48011,"Armstrong County, Texas",011,0.0,0.0,0.1,5.2,4.2,27.2,72750,0.5


    GEOID               NAME           county_fips         pct_hispanic    
 Length:254         Length:254         Length:254         Min.   :0.00000  
 Class :character   Class :character   Class :character   1st Qu.:0.00000  
 Mode  :character   Mode  :character   Mode  :character   Median :0.00000  
                                                          Mean   :0.01417  
                                                          3rd Qu.:0.00000  
                                                          Max.   :0.70000  
   pct_black        pct_white       pct_poverty    pct_uninsured  
 Min.   :0.0000   Min.   : 0.000   Min.   : 4.00   Min.   : 0.00  
 1st Qu.:0.0000   1st Qu.: 0.600   1st Qu.:10.90   1st Qu.:13.90  
 Median :0.2000   Median : 1.000   Median :14.60   Median :16.70  
 Mean   :0.3264   Mean   : 1.255   Mean   :14.89   Mean   :16.97  
 3rd Qu.:0.4750   3rd Qu.: 1.500   3rd Qu.:17.60   3rd Qu.:19.30  
 Max.   :4.9000   Max.   :14.200   Max.   :41.10   Max.   :36.70  

In [10]:
texas_counties <- texas_counties %>%
  mutate(
    County = NAME %>%
      str_to_lower() %>%
      str_remove(" county, texas") %>%
      str_trim() %>%
      str_to_title()
  )
head(texas_counties)

GEOID,NAME,county_fips,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born,County
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
48001,"Anderson County, Texas",001,0.1,0.3,2.6,17.4,18.5,15.4,62068,1.1,Anderson
48003,"Andrews County, Texas",003,0.0,0.1,0.9,15.6,22.4,17.7,72242,1.2,Andrews
48005,"Angelina County, Texas",005,0.0,0.2,3.1,15.8,17.7,17.9,60960,1.2,Angelina
48007,"Aransas County, Texas",007,0.0,0.7,0.9,14.2,12.8,28.8,69466,1.5,Aransas
48009,"Archer County, Texas",009,0.0,0.0,1.5,9.2,13.9,25.4,72159,0.8,Archer
48011,"Armstrong County, Texas",011,0.0,0.0,0.1,5.2,4.2,27.2,72750,0.5,Armstrong


In [11]:
texas_counties <- texas_counties[, !names(texas_counties) %in% c("GEOID", "NAME", "county_fips")]

texas_counties <- texas_counties %>% relocate(County)
head(texas_counties)
write.csv(texas_counties, file = "data/texas_census.csv", row.names = FALSE)

County,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Anderson,0.1,0.3,2.6,17.4,18.5,15.4,62068,1.1
Andrews,0.0,0.1,0.9,15.6,22.4,17.7,72242,1.2
Angelina,0.0,0.2,3.1,15.8,17.7,17.9,60960,1.2
Aransas,0.0,0.7,0.9,14.2,12.8,28.8,69466,1.5
Archer,0.0,0.0,1.5,9.2,13.9,25.4,72159,0.8
Armstrong,0.0,0.0,0.1,5.2,4.2,27.2,72750,0.5


**This section is for further cleaning brfss data as needed**

In [12]:
pct_medical_cost <- read.csv("out/medical_cost.csv") %>%
  select(PHR, group, pct, low, high, answer)
pct_medical_cost <- pct_medical_cost %>%
  filter(group == "Total" & answer == "YES")
pct_medical_cost <- pct_medical_cost[, !names(pct_medical_cost) %in% c("group", "answer", "low", "high")]
pct_medical_cost <- pct_medical_cost %>% rename(medical_cost_pct = pct)
print(pct_medical_cost)

past_flu_shot <- read.csv("out/flu_shot.csv") %>%
  select(PHR, group, pct, low, high, answer)
past_flu_shot <- past_flu_shot %>%
  filter(group == "Total" & answer == "YES")
past_flu_shot <- past_flu_shot[, !names(past_flu_shot) %in% c("group", "answer", "low", "high")]
past_flu_shot <- past_flu_shot %>% rename(flu_shot_pct = pct)
print(past_flu_shot)


   PHR medical_cost_pct
1   10             19.9
2   11             16.9
3    1             20.0
4    2             20.9
5    3             16.6
6    4             21.6
7    5             18.1
8    6             17.3
9    7             13.0
10   8             18.6
11   9             16.8
   PHR flu_shot_pct
1   10         41.1
2   11         32.0
3    1         33.6
4    2         37.4
5    3         38.5
6    4         25.4
7    5         25.6
8    6         36.2
9    7         41.3
10   8         38.0
11   9         24.5


In [14]:
brfss_data <- pct_medical_cost %>%
  inner_join(past_flu_shot, by = "PHR")
print(brfss_data)
write.csv(brfss_data, file = "data/brfss_data.csv", row.names = FALSE)

   PHR medical_cost_pct flu_shot_pct
1   10             19.9         41.1
2   11             16.9         32.0
3    1             20.0         33.6
4    2             20.9         37.4
5    3             16.6         38.5
6    4             21.6         25.4
7    5             18.1         25.6
8    6             17.3         36.2
9    7             13.0         41.3
10   8             18.6         38.0
11   9             16.8         24.5
